# 函数 · 知识点

### 1、函数定义与返回值

In [1]:
# def 定义函数，不用写返回类型（动态类型），没有 return 默认返回 None
def add(a, b):
    return a + b

print(add(1, 2))

def no_return():
    print("没有 return")

result = no_return()
print(result)   # None

# 可以一次返回多个值，本质是返回一个 tuple（02 章学过元组解包）
def min_max(nums):
    return min(nums), max(nums)

lo, hi = min_max([3, 1, 4, 1, 5])
print(lo, hi)

3
没有 return
None
1 5


### 2、位置参数 & 默认参数

In [2]:
# 位置参数：按顺序传值
def greet(name, greeting):
    print(f"{greeting}, {name}")

greet("Tom", "Hello")

# 默认参数：调用时可以省略，用定义时的默认值
def greet2(name, greeting="Hello"):
    print(f"{greeting}, {name}")

greet2("Tom")             # 用默认值
greet2("Tom", "Hi")        # 覆盖默认值

# 注意：默认参数只在函数定义时计算一次，如果默认值是可变对象（list/dict），
# 多次调用会共享同一个对象，这是 02 章讲过的经典陷阱，函数参数里同样适用
def add_item(item, items=None):   # 正确写法：默认值用 None
    if items is None:
        items = []
    items.append(item)
    return items

Hello, Tom
Hello, Tom
Hi, Tom


### 3、关键字参数

In [3]:
# 调用时可以用 参数名=值 的方式传参，不用按位置顺序，可读性更好
def make_user(name, age, city):
    print(f"{name}, {age}岁, 来自{city}")

make_user(name="Tom", city="Beijing", age=18)   # 顺序可以打乱
make_user("Tom", city="Beijing", age=18)        # 位置参数和关键字参数可以混用，但位置参数必须在前

# 强制关键字参数：参数列表里写一个 *，它后面的参数调用时必须用关键字传，不能按位置传
def connect(host, *, port=8080, timeout=30):
    print(host, port, timeout)

connect("localhost", port=9090)   # 必须写 port=
# connect("localhost", 9090)      # 报错！port 不能按位置传

Tom, 18岁, 来自Beijing
Tom, 18岁, 来自Beijing
localhost 9090 30


### 4、*args 和 **kwargs

In [4]:
# *args：收集多余的位置参数，打包成一个 tuple
def total(*args):
    print(args, type(args))
    return sum(args)

print(total(1, 2, 3, 4))

# **kwargs：收集多余的关键字参数，打包成一个 dict
def show_info(**kwargs):
    print(kwargs, type(kwargs))

show_info(name="Tom", age=18)

# 两者可以一起用，顺序固定：普通参数, *args, **kwargs
def func(a, *args, **kwargs):
    print(a, args, kwargs)

func(1, 2, 3, x=4, y=5)

# 调用时反过来用：* 和 ** 把已有的列表/字典"拆包"传进去，这个很常用
nums = [1, 2, 3]
print(total(*nums))         # 等价于 total(1, 2, 3)

info = {"name": "Tom", "age": 18}
show_info(**info)           # 等价于 show_info(name="Tom", age=18)

(1, 2, 3, 4) <class 'tuple'>
10
{'name': 'Tom', 'age': 18} <class 'dict'>
1 (2, 3) {'x': 4, 'y': 5}
(1, 2, 3) <class 'tuple'>
6
{'name': 'Tom', 'age': 18} <class 'dict'>


### 5、变量作用域

In [5]:
# 函数内部定义的变量是局部变量（local），函数外部访问不到
def f():
    x = 10
    print(x)

f()
# print(x)   # 报错，x 是 f 内部的局部变量

# 函数内部可以读取外部（全局）变量，但直接赋值会创建一个新的局部变量，不会修改外部的
count = 0
def increment():
    count = count + 1   # 报错！UnboundLocalError，因为这一行让 count 变成局部变量，
                         # 但赋值号右边又想读取这个还没赋值的局部变量

# 想在函数内部修改全局变量，要显式声明 global
count = 0
def increment_fixed():
    global count
    count += 1

increment_fixed()
print(count)   # 1

# nonlocal：用在嵌套函数里，修改外层函数（不是全局）的变量，闭包场景常用
def outer():
    x = 0
    def inner():
        nonlocal x
        x += 1
        return x
    return inner

counter = outer()
print(counter(), counter(), counter())   # 1 2 3，每次调用 inner 都基于上次的 x

10
1
1 2 3


### 6、lambda 表达式

In [6]:
# lambda 是匿名函数，只能写一个表达式，不能写多行逻辑，常用在需要传一个简单函数的地方
square = lambda x: x * x
print(square(5))

# 等价于：
def square2(x):
    return x * x

# 最常见用途：配合 sorted/map/filter 的 key 或处理函数参数
words = ["banana", "kiwi", "apple", "fig"]
print(sorted(words, key=lambda w: len(w)))   # 按长度排序: ['fig', 'kiwi', 'apple', 'banana']

users = [{"name": "Tom", "age": 18}, {"name": "Jerry", "age": 15}]
print(sorted(users, key=lambda u: u["age"]))  # 按年龄排序

print(list(map(lambda x: x * 2, [1, 2, 3])))       # [2, 4, 6]
print(list(filter(lambda x: x % 2 == 0, range(10))))  # 只保留偶数

25
['fig', 'kiwi', 'apple', 'banana']
[{'name': 'Jerry', 'age': 15}, {'name': 'Tom', 'age': 18}]
[2, 4, 6]
[0, 2, 4, 6, 8]


### 7、函数是一等对象

In [7]:
# 函数本身可以当作值，赋值给变量、当参数传递、当返回值返回
def add(a, b):
    return a + b

def sub(a, b):
    return a - b

# 当参数传递：写一个"高阶函数"，根据传入的函数决定具体行为
def calc(a, b, op):
    return op(a, b)

print(calc(3, 2, add))
print(calc(3, 2, sub))
print(calc(3, 2, lambda x, y: x * y))   # 也可以直接传 lambda

# 当返回值返回：函数可以返回另一个函数（结合第5节的 nonlocal，就是闭包）
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

double = make_multiplier(2)
print(double(5))   # 10

5
1
6
10


### 8、装饰器基础

装饰器本质上就是一个**接受函数、返回新函数**的高阶函数。用 `@` 语法糖写在被装饰函数的上方，等价于 `func = decorator(func)`。

In [8]:
# ============================================================
# 从"高阶函数"到"装饰器"，只有一步之遥
# ============================================================

# 第7节讲过：函数是一等对象，可以当参数传、当返回值返回
# 装饰器就是这种模式的一个特例：接受一个函数，返回一个"增强版"的函数

# ---- 写法一：手动装饰（不用 @）----
def log(func):                          # 参数是一个函数
    def wrapper(*args, **kwargs):       # 用 *args/**kwargs 保证不管原函数参数是什么都能兜住
        print(f"[LOG] 调用 {func.__name__}，参数: {args}, {kwargs}")
        result = func(*args, **kwargs)  # 真正调用原函数
        print(f"[LOG] {func.__name__} 返回: {result}")
        return result
    return wrapper                      # 返回增强后的函数

def add(a, b):
    return a + b

add = log(add)          # 手动装饰：把原函数传给装饰器，用返回值替换原函数
print(add(1, 2))

# ---- 写法二：@ 语法糖（等价于上面的手动装饰）----
@log                     # 这一行等价于 add2 = log(add2)
def add2(a, b):
    return a + b

print(add2(3, 4))       # 调用时自动经过了 log 的增强

# 对比 Java：Java 的 @Override、@Deprecated 只是编译期/运行时的元数据标记，
# 不会改变方法的行为；Python 的 @ 装饰器是在运行时真正替换函数，完全不同</cell_id>


[LOG] 调用 add，参数: (1, 2), {}
[LOG] add 返回: 3
3
[LOG] 调用 add2，参数: (3, 4), {}
[LOG] add2 返回: 7
7


### 9、带参数的装饰器

有时候希望装饰器本身也能接收参数，比如 `@repeat(3)` 让函数执行三次。这时需要**三层嵌套**：最外层接收装饰器的参数，中间层接收函数，最内层接收函数的参数。

In [9]:
# 回顾：普通装饰器（不带参数）是两层嵌套
# def decorator(func):          # 第一层：接收函数
#     def wrapper(*args, **kwargs):  # 第二层：接收函数的参数
#         ...
#     return wrapper

# 带参数的装饰器：三层嵌套
# def decorator_with_args(param):    # 第一层：接收装饰器自己的参数
#     def decorator(func):           # 第二层：接收函数（标准两层结构）
#         def wrapper(*args, **kwargs):  # 第三层：接收函数的参数
#             ...
#         return wrapper
#     return decorator               # 注意：返回的是 decorator

import time

def repeat(n):                         # 第一层：接收装饰器参数 n
    """让被装饰的函数执行 n 次"""
    def decorator(func):               # 第二层：接收函数（标准装饰器结构）
        def wrapper(*args, **kwargs):  # 第三层：接收函数调用的参数
            for i in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)                             # 等价于 greet = repeat(3)(greet)
def greet(name):
    print(f"Hello, {name}!")

greet("Tom")                           # 执行 3 次

# 再举一个实用的例子：带超时判断的装饰器
def max_time(seconds):                 # 第一层：接收超时秒数
    def decorator(func):
        def wrapper(*args, **kwargs):
            start = time.time()
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            if elapsed > seconds:
                print(f"⚠ {func.__name__} 耗时 {elapsed:.2f}s，超过 {seconds}s 上限")
            else:
                print(f"✓ {func.__name__} 耗时 {elapsed:.2f}s")
            return result
        return wrapper
    return decorator

@max_time(0.5)                         # 设置 0.5 秒超时上限
def slow_add(a, b):
    time.sleep(0.3)                    # 模拟耗时操作
    return a + b

slow_add(1, 2)

Hello, Tom!
Hello, Tom!
Hello, Tom!


✓ slow_add 耗时 0.30s


3

### 10、functools.wraps 与内置装饰器简介

装饰器有个小问题：被装饰后，函数的 `__name__`、`__doc__` 等元信息会变成 wrapper 的。`functools.wraps` 可以修复这个问题。另外，Python 内置了一些常用装饰器，如 `@property`、`@staticmethod`、`@classmethod`（第 05 章会详讲）。

In [10]:
# ---- 问题：装饰后函数"丢了名字" ----
def my_decorator(func):
    def wrapper(*args, **kwargs):
        """wrapper 的文档"""
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def say_hello():
    """say_hello 的文档"""
    print("hello")

print(say_hello.__name__)   # wrapper  ← 不是 say_hello！
print(say_hello.__doc__)    # wrapper 的文档 ← 原函数的文档丢了

# ---- 解决：用 functools.wraps ----
import functools

def my_decorator_fixed(func):
    @functools.wraps(func)          # 把 func 的元信息复制到 wrapper 上
    def wrapper(*args, **kwargs):
        """wrapper 的文档"""
        return func(*args, **kwargs)
    return wrapper

@my_decorator_fixed
def say_hi():
    """say_hi 的文档"""
    print("hi")

print(say_hi.__name__)    # say_hi  ← 正确！
print(say_hi.__doc__)     # say_hi 的文档 ← 正确！

# 写装饰器时，养成顺手加 @functools.wraps(func) 的习惯

# ---- 内置装饰器速览（详细在第 05 章）----

# @property：把方法变成"属性"来访问，不用写括号
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

c = Circle(5)
print(c.radius)   # 不用写 c.radius()，像访问属性一样自然

# @staticmethod / @classmethod：定义静态方法和类方法
class Demo:
    @staticmethod
    def util(x):           # 不需要 self
        return x * 2

    @classmethod
    def from_string(cls, s):  # cls 是类本身，不是实例
        return cls()

print(Demo.util(3))  # 直接用类名调用，不需要实例

wrapper
wrapper 的文档
say_hi
say_hi 的文档
5
6
